In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# Parameters (same spirit as ER)
# -----------------------------
n = 100000          # number of nodes
K = 5               # number of blocks (same sized)
M = 97              # number of sampled node-pairs per trial
trials = 1000       # Monte Carlo trials
confidence_level = 0.95

In [ ]:
# --- SBM configuration ---
p_in_values = np.arange(0.2, 1.01, 0.1)   # 0.2 ... 0.8 step 0.1
p_out_values = [0.1, 0.2]

In [ ]:
rng = np.random.default_rng(0)

# ------------------------------------
# Helpers: true density for equal blocks
# ------------------------------------
def true_edge_density_sbm(n: int, K: int, p_in: float, p_out: float) -> float:
    """
    True edge density over all unordered node pairs in an SBM with K equal-sized blocks.
    Uses exact combinatorics (assumes n divisible by K).
    """
    if n % K != 0:
        raise ValueError("n must be divisible by K for equal-sized blocks in this setup.")

    s = n // K
    total_pairs = n * (n - 1) / 2
    within_pairs = K * (s * (s - 1) / 2)
    w_in = within_pairs / total_pairs
    w_out = 1.0 - w_in
    return w_in * p_in + w_out * p_out

# ---------------------------------------------------------
# Estimator: sample M random node-pairs, SBM edge generation
# (keeps the ER structure: average of Bernoulli draws)
# ---------------------------------------------------------
def estimate_edge_density_sbm(n: int, K: int, p_in: float, p_out: float, M: int, rng: np.random.Generator) -> float:
    """
    Estimate edge density by sampling M random unordered node-pairs (u != v),
    then drawing an edge with p_in if within same block, else p_out.
    """
    if n % K != 0:
        raise ValueError("n must be divisible by K for equal-sized blocks in this setup.")
    s = n // K

    edges = np.empty(M, dtype=np.int8)

    for m in range(M):
        # sample two distinct nodes uniformly at random
        u = rng.integers(0, n)
        v = rng.integers(0, n - 1)
        if v >= u:
            v += 1  # ensures v != u while keeping uniformity

        bu = u // s
        bv = v // s
        p = p_in if (bu == bv) else p_out
        edges[m] = (rng.random() < p)

    return edges.mean()

def monte_carlo_sbm(n: int, K: int, p_in: float, p_out: float, M: int, trials: int, rng: np.random.Generator) -> np.ndarray:
    """
    Run Monte Carlo trials and return array of p_hat estimates (length = trials).
    """
    estimates = np.empty(trials, dtype=float)
    for t in range(trials):
        estimates[t] = estimate_edge_density_sbm(n, K, p_in, p_out, M, rng)
    return estimates

# -----------------------------
# Wald CI and coverage (only)
# -----------------------------
def wald_interval(p_hat: float, M: int, z: float) -> tuple[float, float]:
    se = np.sqrt(max(p_hat * (1.0 - p_hat), 0.0) / M)
    return (p_hat - z * se, p_hat + z * se)

def wald_coverage(p_hats: np.ndarray, p_true: float, M: int, z: float) -> float:
    covered = 0
    for p_hat in p_hats:
        lo, hi = wald_interval(p_hat, M, z)
        if lo <= p_true <= hi:
            covered += 1
    return covered / len(p_hats)

In [ ]:
# z for 95% (matching common ER-style usage)
z = 1.96

# ------------------------------------
# Main sweep (close to ER loop structure)
# ------------------------------------
results = {}  # keyed by p_out, each stores arrays over p_in grid

for p_out in p_out_values:
    p_true_list = []
    mean_hat_list = []
    sd_hat_list = []
    coverage_list = []

    for p_in in p_in_values:
        p_true = true_edge_density_sbm(n, K, p_in, p_out)
        p_hats = monte_carlo_sbm(n, K, p_in, p_out, M, trials, rng)

        p_true_list.append(p_true)
        mean_hat_list.append(p_hats.mean())
        sd_hat_list.append(p_hats.std(ddof=1))
        coverage_list.append(wald_coverage(p_hats, p_true, M, z))

        print(
            f"p_out={p_out:.2f}, p_in={p_in:.1f} | "
            f"p_true={p_true:.6f}, mean(p_hat)={p_hats.mean():.6f}, "
            f"bias={p_hats.mean()-p_true:+.6f}, sd={p_hats.std(ddof=1):.6f}, "
            f"Wald cov={coverage_list[-1]:.3f}"
        )

    results[p_out] = {
        "p_in": np.array(p_in_values, dtype=float),
        "p_true": np.array(p_true_list, dtype=float),
        "mean_hat": np.array(mean_hat_list, dtype=float),
        "sd_hat": np.array(sd_hat_list, dtype=float),
        "coverage": np.array(coverage_list, dtype=float),
    }

In [ ]:
# ------------------------------------
# Plot 1: Empirical vs True
# One plot for each p_out (NOT p_in)
# ------------------------------------

dash_styles = ["-", ":"]          # different dashes for the two p_out values
colors = ["blue", "green"]         # matching line + dot colors

for p_out, dash, color in zip(p_out_values, dash_styles, colors):
    plt.figure()

    p_true = results[p_out]["p_true"]
    mean_hat = results[p_out]["mean_hat"]
    sd_hat = results[p_out]["sd_hat"]

    plt.plot(p_true, mean_hat, marker="o", linestyle=dash, color=color, label="Mean estimate")
    plt.fill_between(
        p_true,
        mean_hat - z * sd_hat,
        mean_hat + z * sd_hat,
        alpha=0.2,
        label="95% CI"
    )

    # identity line
    xmin, xmax = p_true.min(), p_true.max()
    plt.plot([xmin, xmax], [xmin, xmax], linestyle="--", linewidth=1, color="orange", label="True global edge density")

    plt.ylim(0, 1)
    plt.xlim(0.1, 0.5)
    plt.xlabel("True global edge density")
    plt.ylabel("Estimated F(G)")
    # plt.title(f"SBM (K=5 equal blocks): Empirical vs True global edge density\np_out = {p_out}")
    plt.legend()
    # plt.grid(True)
    plt.show()


In [ ]:
# ------------------------------------
# Plot 2 (final): Wald coverage vs true p
# Line color matches dot color for each p_in
# ------------------------------------
plt.figure()

dash_styles = ["-", ":"]          # different dashes for the two p_out values
colors = ["blue", "green"]         # matching line + dot colors

for p_out, dash, color in zip(p_out_values, dash_styles, colors):
    plt.plot(
        results[p_out]["p_true"],
        results[p_out]["coverage"],
        linestyle=dash,
        color=color,
        label=f"p_out = {p_out}"
    )
    plt.scatter(
        results[p_out]["p_true"],
        results[p_out]["coverage"],
        color=color
    )

# 95% target line in orange
plt.axhline(
    confidence_level,
    linestyle="--",
    linewidth=1.5,
    color="orange",
    label="Wald 95% CI coverage"
)

plt.ylim(0, 1)
plt.xlim(0.1, 0.5)
plt.xlabel("True global edge density")
plt.ylabel("Coverage probability")
# plt.title("Coverage vs p (SBM, K=5 equal blocks)")
plt.legend()
# plt.grid(True)
plt.show()